# LangChain L4 — Level 3 — Production-shaped tools
OpsPilot v2 works, but its tools would not survive a code review. `calculate` uses `eval`, the
customer tool accepts any string, and descriptions are vague. The lesson of this section:

> **A tool is an API boundary, not just a Python function.**

```text
                 TOOL
                  |
        +---------+----------+
        |                    |
     schema               function
   (what the model      (what your code
    may ask for)         actually does)
        |                    |
   validate args         execute, catch
   describe precisely    errors, return
                         predictable output
```

Good tools have precise descriptions, typed and validated arguments, error *values* instead of
exceptions, predictable output shapes, and a clear read-versus-write classification.

### Step 1 — Remove `eval`: a safe calculator

A model can be talked into asking for `calculate("__import__('os').listdir()")`. We parse the
expression with Python's `ast` module and allow only numbers and arithmetic operators.

In [ ]:
import ast, operator                               # Python standard library; the safe calculator is ours

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"unsupported expression element: {type(node).__name__}")

@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as '127 * 834' or '(500 - 120) / 2'.
    Supports + - * / ** % and parentheses only. Returns the numeric result or an error message."""
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval").body))
    except Exception as exc:
        return f"error: {exc}"          # an error VALUE the model can read and react to

print("normal   :", calculate.invoke({"expression": "(500 - 120) / 2"}))
print("attack   :", calculate.invoke({"expression": "__import__('os').listdir()"}))

### Step 2 — Typed, validated arguments with Pydantic

`args_schema` gives the model a precise interface (field descriptions travel into the prompt)
and gives your code validated input. Invalid ids never reach the database.

In [ ]:
from pydantic import BaseModel, Field, field_validator   # Pydantic: validation library that LangChain uses for schemas

class CustomerLookup(BaseModel):                            # ours: the argument schema
    customer_id: str = Field(description="Customer id in the form 'C' followed by three digits, e.g. 'C001'.")

    @field_validator("customer_id")
    @classmethod
    def check_format(cls, value):
        if not re.fullmatch(r"C\d{3}", value):
            raise ValueError("customer_id must look like C001")
        return value

@tool(args_schema=CustomerLookup)                           # LangChain: validate arguments with our Pydantic schema
def get_customer(customer_id: str) -> str:
    """Retrieve a customer record (name, plan, email, customer since) from the CRM by customer id."""
    record = CUSTOMERS.get(customer_id)
    if record is None:
        return json.dumps({"error": "customer_not_found", "customer_id": customer_id})
    return json.dumps({"id": customer_id, **record})

print("valid   :", get_customer.invoke({"customer_id": "C002"}))
print("missing :", get_customer.invoke({"customer_id": "C999"}))
try:
    get_customer.invoke({"customer_id": "drop table customers"})
except Exception as exc:
    print("invalid :", type(exc).__name__, "- rejected before any code ran")

### Step 3 — Break it: the description is the interface

Give the model a tool called `lookup` described as "Lookup something." and ask a customer
question. Then give it the well-described `get_customer`. Same function body, different behaviour.
Tool descriptions are prompt engineering; treat them as carefully as the system prompt.

In [ ]:
@tool
def lookup(id: str) -> str:
    """Lookup something."""
    return json.dumps(CUSTOMERS.get(id, {"error": "not found"}))

question = "What plan is customer C001 on?"
for label, tools in [("vague tool 'lookup'", [lookup]), ("precise tool 'get_customer'", [get_customer])]:
    agent = create_agent(model=model, tools=tools, system_prompt=OPSPILOT_PROMPT)   # LangChain
    out = agent.invoke({"messages": [{"role": "user", "content": question}]})       # LangGraph
    tools_used = [c["name"] for m in out["messages"] if isinstance(m, AIMessage) for c in m.tool_calls]   # LangChain message attributes
    print(f"{label:28} -> tools used: {str(tools_used or 'none'):22} answer: {text_of(out['messages'][-1])[:90]}")

### Step 4 — Read tools versus write tools

Not all tools are equal. Reading a record is cheap and reversible; refunding money is neither.
We add OpsPilot's first **write tool** now, but it will not be wired to an agent until the
guardrails of L10 and the approval flow of L11 exist. The policy we are heading towards:

```text
get_customer()          -> automatic
search_policies()       -> automatic
send_email()            -> maybe automatic
refund_customer()       -> human approval
delete_customer()       -> human approval, or prohibited
```

In [ ]:
REFUND_LEDGER = []          # ours: every refund ever issued, so we can audit what the agent did

class RefundRequest(BaseModel):
    customer_id: str = Field(description="Customer id, e.g. 'C002'.")
    amount: float = Field(gt=0, le=10000, description="Amount to refund in USD.")
    reason: str = Field(default="duplicate charge", description="Short reason recorded in the ledger.")

@tool(args_schema=RefundRequest)
def refund_customer(customer_id: str, amount: float, reason: str = "duplicate charge") -> str:
    """WRITE ACTION: issue a refund to a customer. Irreversible. Use only after verifying the charge."""
    if customer_id not in CUSTOMERS:
        return json.dumps({"error": "customer_not_found"})
    entry = {"customer_id": customer_id, "amount": amount, "reason": reason, "refund_id": f"R{len(REFUND_LEDGER) + 1:03d}"}
    REFUND_LEDGER.append(entry)
    return json.dumps({"status": "refunded", **entry})

READ_TOOLS = [calculate, get_weather, get_customer, get_order]
WRITE_TOOLS = [refund_customer]
print("read  tools:", [t.name for t in READ_TOOLS])
print("write tools:", [t.name for t in WRITE_TOOLS], "(not yet given to any agent)")

### Recap

- **Problem seen:** `eval`, untyped arguments and vague descriptions make tools unsafe and unreliable.
- **Layer added:** a parsed calculator, Pydantic `args_schema`, error values, and a read/write classification.
- **Evidence:** the injection expression was rejected, the bad id never reached the CRM, and the vague tool was not used.